# Sampler sigma_max and step spacing

**Prof. Baptista's ask 2, verbatim:** *"the sigma_max/spacing mismatch with the paper is worth
matching (or at least testing these spacings) if its easy since it could shift the point the
model collapses to."*

Both knobs are **inference-time only**. In EDM the training noise level is drawn from a
lognormal set by `P_mean` and `P_std`, independent of the sampler's `sigma_max`; the sampler
grid is only used when generating. So one trained model can be re-sampled under every cell of
the grid, and every difference measured is attributable to the sampler and nothing else. This
is the same design as the earlier `edm_unet_train_size_sweep_sigma_fix.ipynb` ablation.

## The two mismatches, separately

Section 1 of `notes/results_table.md` decomposes them, and they are not the same kind of thing.

| | Baptista et al. | this project | factor |
|---|---|---|---|
| backward steps | **40** (their section 5.4, stated) | 1000 | 25x |
| `sigma_max` | 80 (assumed; the string never appears in their paper) | 10 | 8x |
| `sigma_max / D_-` | ~2.9 (rests on both assumptions) | 0.05 to 0.08 | ~40x |

The step count is the only one of the three that is stated in their paper rather than inferred.
The `sigma_max/D_-` ratio additionally depends on their square side length, which they also do
not state, so the ~2.9 target carries two layers of assumption and the sweep should not be read
as "matching their setup" so much as spanning the range that contains it.

## Grid

`sigma_max` in {10, 80, 400} x `n_steps` in {40, 1000}, at `n_train` in {2, 8}.

- `sigma_max = 10` is this project's locked convention; `80` is the value assumed of theirs;
  `400` is the order needed to bring `sigma_max/D_min` near their ~2.9 on this dataset.
- `n_steps = 40` is their stated count; `1000` is this project's.
- `n_train = 2` is the most memorization-prone setting and the one they use; `8` is this
  project's workhorse and the setting of Results D and E.

Twelve U-Net cells and twelve closed-form cells. The closed-form score travels with every cell
as a positive control, so a null in any cell can be distinguished from a broken sampler.

**This supersedes the earlier ablation**, which tested `sigma_max` in {10, 80} x steps in
{500, 2000}. That grid was null, and arguably null by construction: both `sigma_max` values sit
at 0.05 and 0.55 of `D_min`, the same regime, and neither step count was near their 40.

**Decision rule, fixed before the numbers are seen.** The result is a shift in the sampler's
behaviour only if some cell moves the coarse gap below the neutral line by more than the seed
spread measured in `capacity_multiseed.ipynb`, or raises the pixel collapse fraction above 0.
A change smaller than that is not a shift. If nothing moves, the write-up says the sampler was
ruled out across this range, not that a mismatch was diagnosed or fixed.


In [1]:
import sys, os, math, time, copy
import numpy as np
import torch
import matplotlib.pyplot as plt

# -- path setup --
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
src_dir = os.path.join(repo_root, 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

import diffusion_score_models as score_models
from multiband_data_utils import generate_multiband_dataset_postmask
from memorization_metrics import RingMetricContext
from edm import EDMPrecond, EDMScoreWrapper, train_edm
from unet import SmallUNet, count_parameters
from device_utils import resolve_device

DEVICE = resolve_device()   # cuda > mps > cpu; use resolve_device("cpu") on the personal laptop
print(f'device: {DEVICE}')

device: cuda


In [2]:
# -- Data generation (identical config to the earlier memorization notebooks) --
components = [
    {"name": "coarse", "length_scale": 2.0,  "s": 2.0, "sigma_sq": 1.0, "band": (0.5, 4.0)},
    {"name": "mid1",   "length_scale": 6.0,  "s": 2.0, "sigma_sq": 1.0, "band": (4.0, 10.0)},
    {"name": "mid2",   "length_scale": 12.0, "s": 2.0, "sigma_sq": 1.0, "band": (10.0, 18.0)},
    {"name": "fine",   "length_scale": 24.0, "s": 2.0, "sigma_sq": 1.0, "band": (18.0, 32.0)},
]
result = generate_multiband_dataset_postmask(
    num_samples=200, grid_size=128, components=components,
    weights=[1.0, 0.8, 0.8, 1.2], seed=42, normalize=True,
)
bands = result.get('bands', {c['name']: c['band'] for c in components})
N = 128
x_all = result['combined']          # kept on CPU; slices moved to DEVICE as needed
ctx = RingMetricContext(N, bands, device=DEVICE)
print(f'x_all: {tuple(x_all.shape)}')

x_all: (200, 128, 128)


In [3]:
# -- Evaluation: matched sampler (sigma_max=10, config D of the sigma-fix study) --
VE_SAMPLE = score_models.VE_EDM(sigma_min=0.002, sigma_max=10.0)
N_GEN = 16
N_SDE_STEPS = 1000
N_RAND_REF = 32
LATENT_SEED = 42

@torch.no_grad()
def sample_from(score_fn):
    torch.manual_seed(LATENT_SEED)   # same latents (and step noise stream) for every run
    latents = torch.randn(N_GEN, N*N, device=DEVICE)
    out = VE_SAMPLE.SDEsampler(score_fn, latents, num_steps=N_SDE_STEPS)
    return out.reshape(N_GEN, N, N)

@torch.no_grad()
def pixel_nn_stats(x_gen, x_train):
    '''Relative pixel-space L2 distance to the nearest training field (Baptista-style
    collapse measure). Returns per-sample distances; threshold at plot time.'''
    d = torch.cdist(x_gen.flatten(1), x_train.flatten(1))
    nn_rel = d.min(dim=1).values / x_train.flatten(1).norm(dim=1).mean()
    return nn_rel.cpu()

@torch.no_grad()
def evaluate_checkpoint(precond, x_train):
    wrapper = EDMScoreWrapper(precond, VE_SAMPLE.marginal_prob_std, N, c_tikhonov=0.0).to(DEVICE)
    x_gen = sample_from(wrapper)
    m = ctx.evaluate(x_gen, x_train, n_rand_ref=N_RAND_REF, exclude_nn=True)
    return {
        'coarse_score': m['coarse_score'].mean().item(),
        'fine_score': m['fine_score'].mean().item(),
        'mean_ratio': m['mean_ratio'].cpu(),
        'nn_rel': pixel_nn_stats(x_gen, x_train),
        'samples': x_gen[:2].cpu(),
    }

@torch.no_grad()
def gmm_reference(x_train):
    train_flat = x_train.reshape(x_train.shape[0], -1)
    gmm = score_models.GMM_score(train_flat, VE_SAMPLE.marginal_prob_mean,
                                 VE_SAMPLE.marginal_prob_std)
    x_gen = sample_from(gmm)
    m = ctx.evaluate(x_gen, x_train, n_rand_ref=N_RAND_REF, exclude_nn=True)
    return {
        'coarse_score': m['coarse_score'].mean().item(),
        'fine_score': m['fine_score'].mean().item(),
        'nn_rel': pixel_nn_stats(x_gen, x_train),
    }

results_dir = os.path.join(repo_root, 'results', 'data')
os.makedirs(results_dir, exist_ok=True)
fig_dir = os.path.join(repo_root, 'results', 'figures')
os.makedirs(fig_dir, exist_ok=True)

In [4]:
# -- Grid. Sampler knobs only; nothing here touches training. --
SMOKE = False

SIGMA_MAX_VALUES = [10.0, 80.0, 400.0]   # locked convention / assumed theirs / matched to D_-
N_STEPS_VALUES   = [40, 1000]            # theirs (stated) / this project's
N_TRAIN_VALUES   = [2, 8]

TOTAL_STEPS  = 30000                     # identical to the transition sweep
EVAL_AT      = 30000                     # fully-trained checkpoint only
BATCH_SIZE   = 8
TRAIN_SEED   = 0

if SMOKE:
    TOTAL_STEPS = 200; EVAL_AT = 200
    SIGMA_MAX_VALUES = [10.0, 400.0]; N_STEPS_VALUES = [40]; N_TRAIN_VALUES = [2]
    N_GEN = 4

ckpt_path = os.path.join(results_dir, 'sigma_spacing_checkpoints.pt')

# Where each cell sits on the dimensionless axis of Assumption 4.1.
print(f"{'n_train':>8} {'D_min':>8} " + " ".join(f"{'s='+str(int(s)):>9}" for s in SIGMA_MAX_VALUES))
D_MIN = {}
for n in N_TRAIN_VALUES:
    f = x_all[:n].flatten(1)
    D = torch.cdist(f, f); D.fill_diagonal_(float('inf'))
    D_MIN[n] = D.min().item()
    print(f"{n:>8} {D_MIN[n]:>8.1f} " + " ".join(f"{s/D_MIN[n]:>9.3f}" for s in SIGMA_MAX_VALUES))
print("\n(Baptista et al. sit at roughly 2.9 on this axis, under the two assumptions above.)")
print(f"\n{len(SIGMA_MAX_VALUES)*len(N_STEPS_VALUES)*len(N_TRAIN_VALUES)} cells, "
      f"U-Net and closed form each. Training runs needed: {len(N_TRAIN_VALUES)}")
print(f"device: {DEVICE}")


 n_train    D_min      s=10      s=80     s=400
       2    185.9     0.054     0.430     2.152
       8    146.7     0.068     0.545     2.727

(Baptista et al. sit at roughly 2.9 on this axis, under the two assumptions above.)

12 cells, U-Net and closed form each. Training runs needed: 2
device: cuda


In [5]:
# -- Sampler factory: the only thing that varies across cells --
@torch.no_grad()
def sample_with(score_fn_factory, sigma_max, n_steps):
    """Sample under a given (sigma_max, n_steps). Latents are redrawn from the same seed for
    every cell, so all cells share the initial noise; the step-noise stream necessarily differs
    when n_steps differs, which is intrinsic to changing the discretization."""
    ve = score_models.VE_EDM(sigma_min=0.002, sigma_max=sigma_max)
    score_fn = score_fn_factory(ve)
    torch.manual_seed(LATENT_SEED)
    latents = torch.randn(N_GEN, N * N, device=DEVICE)
    out = ve.SDEsampler(score_fn, latents, num_steps=n_steps)
    return out.reshape(N_GEN, N, N)

def unet_factory(precond):
    return lambda ve: EDMScoreWrapper(precond, ve.marginal_prob_std, N, c_tikhonov=0.0).to(DEVICE)

def gmm_factory(train_flat):
    return lambda ve: score_models.GMM_score(train_flat, ve.marginal_prob_mean,
                                             ve.marginal_prob_std)

@torch.no_grad()
def score_cell(x_gen, x_train):
    m = ctx.evaluate(x_gen, x_train, n_rand_ref=N_RAND_REF, exclude_nn=True)
    return {'coarse_score': m['coarse_score'].mean().item(),
            'fine_score': m['fine_score'].mean().item(),
            'mean_ratio': m['mean_ratio'].cpu(),
            'nn_rel': pixel_nn_stats(x_gen, x_train),
            'samples': x_gen[:2].cpu()}

@torch.no_grad()
def neutral_for(x_train, n_seeds=5):
    hold = x_all[100:100+N_GEN].to(DEVICE)
    cs = []
    for s in range(n_seeds):
        torch.manual_seed(s)
        cs.append(ctx.evaluate(hold, x_train, n_rand_ref=N_RAND_REF,
                               exclude_nn=True)['coarse_score'].mean().item())
    return float(np.mean(cs)), float(np.std(cs))


In [6]:
# -- Training: one model per n_train, seed 0, identical to the transition sweep.
#    Reuses edm_unet_transition_checkpoints.pt when it is on disk, else trains and banks. --
all_ckpts = {}
if os.path.exists(ckpt_path):
    all_ckpts = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    print(f'resuming: {sorted(all_ckpts.keys())}')

legacy = os.path.join(results_dir, 'edm_unet_transition_checkpoints.pt')
if os.path.exists(legacy):
    old = torch.load(legacy, map_location='cpu', weights_only=False)
    for n in N_TRAIN_VALUES:
        if n not in all_ckpts and n in old and EVAL_AT in old[n]:
            all_ckpts[n] = old[n][EVAL_AT]
            print(f'n_train={n}: reusing transition-sweep checkpoint at step {EVAL_AT}')

for n in N_TRAIN_VALUES:
    if n in all_ckpts:
        continue
    print(f'===== training n_train={n} (seed {TRAIN_SEED}, {TOTAL_STEPS} steps) =====')
    t0 = time.time()
    saved = train_edm(x_all[:n].reshape(n, -1).to(DEVICE), grid_size=N,
                      total_steps=TOTAL_STEPS, checkpoint_at=[EVAL_AT],
                      base_channels=16, emb_dim=64, lr=1e-3, batch_size=BATCH_SIZE,
                      seed=TRAIN_SEED, device=DEVICE, UNetClass=SmallUNet)
    p = saved[EVAL_AT]
    all_ckpts[n] = {'state_dict': {k: v.cpu() for k, v in p.state_dict().items()},
                    'sigma_data': p.sigma_data}
    torch.save(all_ckpts, ckpt_path)
    print(f'  {time.time()-t0:.0f}s; saved -> {ckpt_path}')


===== training n_train=2 (seed 0, 30000 steps) =====
Estimated sigma_data = 1.0521


  step 1/30000  loss=0.940116


  step 500/30000  loss=0.160906


  step 1000/30000  loss=0.068578


  step 1500/30000  loss=0.027819


...

  step 28000/30000  loss=0.048998


  step 28500/30000  loss=0.033389


  step 29000/30000  loss=0.030866


  step 29500/30000  loss=0.027483


  step 30000/30000  loss=0.020712
  >> checkpoint saved at step 30000
  743s; saved -> /project/6109267/ishiyer/research-diffusion/results/data/sigma_spacing_checkpoints.pt


In [7]:
# -- The sweep: same model, same latent seed, twelve sampler settings --
results, neutrals = {}, {}

for n in N_TRAIN_VALUES:
    x_train = x_all[:n].to(DEVICE)
    train_flat = x_train.reshape(n, -1)
    nb_, ns_ = neutral_for(x_train)
    neutrals[n] = (nb_, ns_)
    print(f"\n########## n_train = {n}   neutral coarse {nb_:.4f} +/- {ns_:.4f} "
          f"  D_min {D_MIN[n]:.1f} ##########")

    unet = SmallUNet(base_channels=16, emb_dim=64, num_levels=3).to(DEVICE)
    precond = EDMPrecond(unet, sigma_data=all_ckpts[n]['sigma_data']).to(DEVICE)
    precond.load_state_dict(all_ckpts[n]['state_dict']); precond.eval()

    for sm in SIGMA_MAX_VALUES:
        for ns in N_STEPS_VALUES:
            t0 = time.time()
            xg_u = sample_with(unet_factory(precond), sm, ns)
            xg_g = sample_with(gmm_factory(train_flat), sm, ns)
            ru, rg = score_cell(xg_u, x_train), score_cell(xg_g, x_train)
            results[(n, sm, ns)] = {'unet': ru, 'gmm': rg,
                                    'sigma_over_dmin': sm / D_MIN[n]}
            print(f"  sigma_max={sm:>5.0f} steps={ns:>5}  "
                  f"unet coarse={ru['coarse_score']:.4f} gap={ru['coarse_score']-nb_:+.4f} "
                  f"collapse={(ru['nn_rel']<0.3).float().mean():.2f}  |  "
                  f"gmm coarse={rg['coarse_score']:.5f} "
                  f"collapse={(rg['nn_rel']<0.3).float().mean():.2f}  ({time.time()-t0:.0f}s)")

torch.save({'results': results, 'neutrals': neutrals, 'D_min': D_MIN,
            'sigma_max_values': SIGMA_MAX_VALUES, 'n_steps_values': N_STEPS_VALUES,
            'n_train_values': N_TRAIN_VALUES, 'eval_at': EVAL_AT,
            'train_seed': TRAIN_SEED, 'device': str(DEVICE),
            'note': ('Sampler sigma_max x step-count sweep on fixed trained models. '
                     'Inference-only: EDM training noise is drawn from the P_mean/P_std '
                     'lognormal and is independent of sampler sigma_max, so every difference '
                     'here is attributable to the sampler. Supersedes '
                     'edm_unet_train_size_sweep_sigma_fix.ipynb.')},
           os.path.join(results_dir, 'sigma_spacing_sweep_v2.pt'))
print('\nsaved -> sigma_spacing_sweep_v2.pt')



########## n_train = 2   neutral coarse 0.8567 +/- 0.0000   D_min 185.9 ##########


  sigma_max=   10 steps=   40  unet coarse=0.8979 gap=+0.0412 collapse=0.00  |  gmm coarse=0.00007 collapse=1.00  (0s)


  sigma_max=   10 steps= 1000  unet coarse=0.8600 gap=+0.0033 collapse=0.00  |  gmm coarse=0.00007 collapse=1.00  (4s)


  sigma_max=   80 steps=   40  unet coarse=0.8160 gap=-0.0407 collapse=0.00  |  gmm coarse=0.00007 collapse=1.00  (0s)


  sigma_max=   80 steps= 1000  unet coarse=0.8294 gap=-0.0273 collapse=0.00  |  gmm coarse=0.00007 collapse=1.00  (4s)


...

  sigma_max=   10 steps= 1000  unet coarse=0.8397 gap=-0.0080 collapse=0.00  |  gmm coarse=0.00008 collapse=1.00  (4s)


  sigma_max=   80 steps=   40  unet coarse=0.8333 gap=-0.0144 collapse=0.00  |  gmm coarse=0.00007 collapse=1.00  (0s)


  sigma_max=   80 steps= 1000  unet coarse=0.8445 gap=-0.0032 collapse=0.00  |  gmm coarse=0.00008 collapse=1.00  (4s)


  sigma_max=  400 steps=   40  unet coarse=0.8490 gap=+0.0013 collapse=0.00  |  gmm coarse=0.00007 collapse=1.00  (0s)


  sigma_max=  400 steps= 1000  unet coarse=0.8425 gap=-0.0052 collapse=0.00  |  gmm coarse=0.00008 collapse=1.00  (4s)

saved -> sigma_spacing_sweep_v2.pt


In [8]:
# -- Summary against the decision rule --
print("gap = U-Net coarse - neutral coarse for that n_train. Negative = toward copying.\n")
for n in N_TRAIN_VALUES:
    nb_, ns_ = neutrals[n]
    print(f"n_train = {n}   neutral {nb_:.4f} +/- {ns_:.4f}")
    print(f"  {'sigma_max':>10} {'s/D_min':>8} " +
          " ".join(f"{'steps='+str(s):>13}" for s in N_STEPS_VALUES))
    for sm in SIGMA_MAX_VALUES:
        cells = []
        for ns in N_STEPS_VALUES:
            r = results[(n, sm, ns)]['unet']
            cells.append(f"{r['coarse_score']-nb_:+.4f}/{(r['nn_rel']<0.3).float().mean():.2f}")
        print(f"  {sm:>10.0f} {sm/D_MIN[n]:>8.3f} " + " ".join(f"{c:>13}" for c in cells))
    print("  (gap / pixel collapse fraction)")

gaps = {k: v['unet']['coarse_score'] - neutrals[k[0]][0] for k, v in results.items()}
coll = {k: (v['unet']['nn_rel'] < 0.3).float().mean().item() for k, v in results.items()}
gcol = {k: (v['gmm']['nn_rel'] < 0.3).float().mean().item() for k, v in results.items()}

worst = min(gaps, key=gaps.get)
print(f"\nLargest movement toward copying: n_train={worst[0]}, sigma_max={worst[1]:.0f}, "
      f"steps={worst[2]}  gap {gaps[worst]:+.4f}")
print(f"Range of gaps across all {len(gaps)} cells: {min(gaps.values()):+.4f} to {max(gaps.values()):+.4f} "
      f"(spread {max(gaps.values())-min(gaps.values()):.4f})")
print(f"Max U-Net pixel collapse fraction over all cells: {max(coll.values()):.4f}")
print(f"Min closed-form collapse fraction over all cells: {min(gcol.values()):.2f} "
      f"<- positive control; should be 1.00 in every cell")

if min(gcol.values()) < 0.95:
    bad = [k for k, v in gcol.items() if v < 0.95]
    print(f"\n  WARNING: the closed-form control fails to collapse in {len(bad)} cell(s): {bad}")
    print("  Those cells measure a broken sampler, not a property of the network. "
          "Do not read the U-Net numbers there.")

print("\nAgainst the pre-registered rule:")
print(f"  measured spread across the whole grid : {max(gaps.values())-min(gaps.values()):.4f}")
print( "  seed spread from capacity_multiseed   : fill in once that notebook has run")
print( "  any nonzero U-Net collapse            : "
      + ("YES -> a genuine shift" if max(coll.values()) > 0 else "no"))


gap = U-Net coarse - neutral coarse for that n_train. Negative = toward copying.

n_train = 2   neutral 0.8567 +/- 0.0000
   sigma_max  s/D_min      steps=40    steps=1000
          10    0.054  +0.0412/0.00  +0.0033/0.00
          80    0.430  -0.0407/0.00  -0.0273/0.00
         400    2.152  -0.0665/0.00  +0.0006/0.00
  (gap / pixel collapse fraction)
n_train = 8   neutral 0.8477 +/- 0.0064
   sigma_max  s/D_min      steps=40    steps=1000
          10    0.068  -0.0187/0.00  -0.0080/0.00
          80    0.545  -0.0144/0.00  -0.0032/0.00
         400    2.727  +0.0013/0.00  -0.0052/0.00
  (gap / pixel collapse fraction)

Largest movement toward copying: n_train=2, sigma_max=400, steps=40  gap -0.0665
Range of gaps across all 12 cells: -0.0665 to +0.0412 (spread 0.1077)
Max U-Net pixel collapse fraction over all cells: 0.0000
Min closed-form collapse fraction over all cells: 1.00 <- positive control; should be 1.00 in every cell

Against the pre-registered rule:
  measured spread acro

In [9]:
# -- Plot: gap and collapse against the dimensionless axis --
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
marks = {40: 'o', 1000: 's'}
cols = {2: 'tab:blue', 8: 'tab:orange'}

for ax, key, lab in [(axes[0], 'gap', 'gap from neutral (coarse)'),
                     (axes[1], 'collapse', 'pixel collapse fraction')]:
    for n in N_TRAIN_VALUES:
        for ns in N_STEPS_VALUES:
            xs = [results[(n, sm, ns)]['sigma_over_dmin'] for sm in SIGMA_MAX_VALUES]
            ys = ([gaps[(n, sm, ns)] for sm in SIGMA_MAX_VALUES] if key == 'gap'
                  else [coll[(n, sm, ns)] for sm in SIGMA_MAX_VALUES])
            ax.plot(xs, ys, marker=marks[ns], color=cols[n], ms=6,
                    ls='-' if ns == 1000 else '--',
                    label=f'n={n}, {ns} steps')
    ax.axvline(2.9, color='red', lw=1.0, ls=':', label='Baptista et al. (assumed)')
    ax.set_xscale('log'); ax.set_xlabel('sigma_max / D_min')
    ax.set_ylabel(lab); ax.legend(fontsize=7)
axes[0].axhline(0.0, color='black', lw=1.0, ls=':')
axes[1].set_ylim(-0.02, 1.02)
axes[0].set_title('does the sampler move the U-Net toward the training set?')
axes[1].set_title('does any cell produce collapse? (closed form gives 1.00)')
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, 'sigma_spacing_sweep.png'), dpi=150, bbox_inches='tight')
plt.show()


## Reading this

The pre-registered rule is at the top. Applying it:

- **A cell with nonzero U-Net pixel collapse fraction is a genuine shift**, and the interesting
  one, because collapse is binary and needs no threshold. The closed-form score collapses at
  1.00 in every cell by construction; the U-Net has been at 0.00 everywhere in this project
  except `d12` in the dimension probe.
- **A change in the coarse gap smaller than the measured seed spread is not a result.** That
  spread comes from `capacity_multiseed.ipynb` and has to be filled in from there; it is
  deliberately not hardcoded here, because the assumed 0.03 to 0.06 used elsewhere in this
  project was never measured.
- **If nothing moves anywhere on the grid**, the correct statement is that the sampler was
  ruled out across `sigma_max/D_min` from 0.05 to about 3, spanning the value inferred for
  Baptista et al., at both 40 and 1000 steps. Ruled out, not diagnosed and not fixed. The
  earlier ablation was written up as a diagnosis once already and had to be corrected.

One asymmetry to keep in mind when reading a null here. Changing `n_steps` necessarily changes
the step-noise stream, so cells at 40 and 1000 steps do not share their stochastic path even
though they share the initial latents. Differences between step counts therefore carry sampling
noise that differences between `sigma_max` values at fixed `n_steps` do not.
